# פרויקט מסכם — מערכת סוכנים לתשלומים דמוית Bit
## מערכות בינה מלאכותית — Agentic AI

במחברת זו נבנה מערכת סוכנים מלאה המנהלת סימולציה של מערכת תשלומים דמוית Bit.
המערכת מורכבת ממספר סוכנים הפועלים יחד: סוכן ניתוב, סוכן תזמור, סוכני תשלום, סוכן זיהוי הונאות, סוכן אבטחה, סוכן הסבר, סוכן ביקורת, סוכן מדיניות וסוכן גיבוי.

> **בסיס:** הפרויקט מבוסס על תרגיל 4 ומשתמש באותו מבנה `AgentResult`, מנגנון ניתוב, בחירת כלים, זיכרון קצר-טווח וסוכן ביקורת — עם הרחבה משמעותית לתחום התשלומים.

### אלמנטים מתקדמים שמומשו
1. **שמירה וטעינה מ-JSON** — שמירת כל המשתמשים, הארנקים והעסקאות לקובץ JSON וטעינתם חזרה.
2. **PolicyAgent** — סוכן מדיניות האוכף מגבלות עסקיות כמו סכום מקסימלי להעברה ביום.

## חלק א׳ — הכנת סביבת העבודה

נתקין ונייבא את כל התלויות הנדרשות. מפתח ה-API של OpenAI נטען מקובץ `.env` באמצעות `python-dotenv`.
אם המפתח חסר או שהקריאה ל-API נכשלת, המערכת תיפול חזרה לתשובה מקומית חלופית.

In [ ]:
# Cell 1 - imports and environment setup
import json
import re
import os
import uuid
import datetime
from dataclasses import dataclass, field, asdict
from typing import Dict, Any, Optional, List

from dotenv import load_dotenv
load_dotenv(override=True)

# OpenAI setup
from openai import OpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
openai_client = None
if OPENAI_API_KEY:
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    print("✅ OpenAI API key loaded successfully.")
else:
    print("⚠️ OPENAI_API_KEY not found in .env — will use local fallback answers.")

✅ OpenAI API key loaded successfully.


## חלק ב׳ — מבנה אחיד לתוצאות סוכנים

כל סוכן במערכת מחזיר אובייקט `AgentResult` אחיד הכולל את שם הסוכן, התוצאה, רמת הביטחון ומטא-דאטה נוספת. מבנה אחיד זה מאפשר לסוכן התזמור לעבוד עם כל סוכן בצורה זהה.

In [ ]:
# Cell 2 - shared result object
@dataclass
class AgentResult:
    agent_name: str
    output: Any
    confidence: float = 1.0
    metadata: Optional[Dict[str, Any]] = None

## חלק ג׳ — מערכת התשלומים

מערכת התשלומים היא הליבה העסקית של הפרויקט. היא כוללת ארבע ישויות:
- **משתמש (User):** מזהה, שם ומספר טלפון.
- **ארנק (Wallet):** מזהה משתמש ויתרה.
- **עסקה (Transaction):** שולח, מקבל, סכום, זמן, סטטוס וציון סיכון.
- **בקשת תשלום (PaymentRequest):** מבקש, משלם, סכום, סטטוס וזמן יצירה.

### כללים עסקיים חובה
| כלל | תיאור |
|---|---|
| יתרה התחלתית | לא ניתן ליצור משתמש עם יתרה שלילית |
| סכום העברה | לא ניתן להעביר סכום שלילי או אפס |
| מקבל | לא ניתן להעביר כסף למשתמש שאינו קיים |
| העברה עצמית | לא ניתן להעביר כסף לעצמך |
| יתרה מספקת | לא ניתן להעביר ללא יתרה מספקת |
| כפל אישור | לא ניתן לאשר/לדחות בקשת תשלום שכבר טופלה |
| היסטוריה | כל העברה נשמרת בהיסטוריית העסקאות |
| לוג | כל פעולה משמעותית נרשמת ב-audit_log |

In [ ]:
# Cell 3 - Payment System entities and logic

@dataclass
class User:
    user_id: str
    name: str
    phone_number: str

@dataclass
class Wallet:
    user_id: str
    balance: float

@dataclass
class Transaction:
    tx_id: str
    sender_id: str
    receiver_id: str
    amount: float
    timestamp: str
    status: str       # "completed", "failed"
    risk_score: float = 0.0

@dataclass
class PaymentRequest:
    request_id: str
    requester_id: str
    payer_id: str
    amount: float
    status: str       # "pending", "approved", "rejected"
    created_at: str


class PaymentSystem:
    """Core payment system — implements all business rules."""

    def __init__(self):
        self.users: Dict[str, User] = {}
        self.wallets: Dict[str, Wallet] = {}
        self.transactions: List[Transaction] = []
        self.payment_requests: Dict[str, PaymentRequest] = {}
        self.audit_log: List[Dict[str, Any]] = []

    def _log(self, action: str, details: dict):
        self.audit_log.append({
            "action": action,
            "details": details,
            "timestamp": datetime.datetime.now().isoformat()
        })

    def _ts(self) -> str:
        return datetime.datetime.now().isoformat()

    # ── create_user ───────────────────────────────────────
    def create_user(self, name: str, phone_number: str, initial_balance: float = 0.0) -> dict:
        if initial_balance < 0:
            self._log("create_user_failed", {"reason": "negative initial balance", "name": name})
            return {"success": False, "error": "לא ניתן ליצור משתמש עם יתרה התחלתית שלילית."}
        user_id = uuid.uuid4().hex[:8]
        self.users[user_id] = User(user_id, name, phone_number)
        self.wallets[user_id] = Wallet(user_id, initial_balance)
        self._log("create_user", {"user_id": user_id, "name": name, "balance": initial_balance})
        return {"success": True, "user_id": user_id, "name": name, "balance": initial_balance}

    # ── get_balance ───────────────────────────────────────
    def get_balance(self, user_id: str) -> dict:
        if user_id not in self.users:
            return {"success": False, "error": "משתמש לא נמצא."}
        return {"success": True, "user_id": user_id, "balance": self.wallets[user_id].balance}

    # ── transfer_money ────────────────────────────────────
    def transfer_money(self, sender_id: str, receiver_id: str, amount: float) -> dict:
        if amount <= 0:
            self._log("transfer_failed", {"reason": "non-positive amount", "amount": amount})
            return {"success": False, "error": "לא ניתן להעביר סכום שלילי או אפס."}
        if sender_id not in self.users:
            self._log("transfer_failed", {"reason": "sender not found", "sender": sender_id})
            return {"success": False, "error": "השולח לא נמצא במערכת."}
        if receiver_id not in self.users:
            self._log("transfer_failed", {"reason": "receiver not found", "receiver": receiver_id})
            return {"success": False, "error": "המקבל לא נמצא במערכת."}
        if sender_id == receiver_id:
            self._log("transfer_failed", {"reason": "self-transfer", "user": sender_id})
            return {"success": False, "error": "לא ניתן להעביר כסף לעצמך."}
        if self.wallets[sender_id].balance < amount:
            self._log("transfer_failed", {"reason": "insufficient funds",
                                          "balance": self.wallets[sender_id].balance, "amount": amount})
            return {"success": False, "error": "אין יתרה מספקת."}

        # execute transfer
        self.wallets[sender_id].balance -= amount
        self.wallets[receiver_id].balance += amount
        tx = Transaction(
            tx_id=uuid.uuid4().hex[:8],
            sender_id=sender_id,
            receiver_id=receiver_id,
            amount=amount,
            timestamp=self._ts(),
            status="completed"
        )
        self.transactions.append(tx)
        self._log("transfer_money", {
            "tx_id": tx.tx_id, "from": sender_id,
            "to": receiver_id, "amount": amount
        })
        return {
            "success": True, "tx_id": tx.tx_id,
            "amount": amount,
            "sender_balance": self.wallets[sender_id].balance,
            "receiver_balance": self.wallets[receiver_id].balance
        }

    # ── get_transactions ─────────────────────────────────
    def get_transactions(self, user_id: str) -> dict:
        if user_id not in self.users:
            return {"success": False, "error": "משתמש לא נמצא."}
        txs = [asdict(t) for t in self.transactions
               if t.sender_id == user_id or t.receiver_id == user_id]
        return {"success": True, "user_id": user_id, "transactions": txs}

    # ── request_payment ──────────────────────────────────
    def request_payment(self, requester_id: str, payer_id: str, amount: float) -> dict:
        if requester_id not in self.users:
            return {"success": False, "error": "המבקש לא נמצא במערכת."}
        if payer_id not in self.users:
            return {"success": False, "error": "המשלם לא נמצא במערכת."}
        if amount <= 0:
            return {"success": False, "error": "סכום הבקשה חייב להיות חיובי."}
        req_id = uuid.uuid4().hex[:8]
        req = PaymentRequest(req_id, requester_id, payer_id, amount, "pending", self._ts())
        self.payment_requests[req_id] = req
        self._log("request_payment", {"request_id": req_id, "requester": requester_id,
                                       "payer": payer_id, "amount": amount})
        return {"success": True, "request_id": req_id, "amount": amount, "status": "pending"}

    # ── approve_payment_request ────────────────────────────
    def approve_payment_request(self, request_id: str) -> dict:
        if request_id not in self.payment_requests:
            return {"success": False, "error": "בקשת תשלום לא נמצאה."}
        req = self.payment_requests[request_id]
        if req.status != "pending":
            self._log("approve_failed", {"request_id": request_id, "current_status": req.status})
            return {"success": False,
                    "error": f"לא ניתן לאשר בקשה שסטטוס שלה כבר: {req.status}"}
        result = self.transfer_money(req.payer_id, req.requester_id, req.amount)
        if result["success"]:
            req.status = "approved"
            self._log("approve_payment", {"request_id": request_id})
            return {"success": True, "request_id": request_id, "transfer": result}
        else:
            return {"success": False, "error": result["error"]}

    # ── reject_payment_request ─────────────────────────────
    def reject_payment_request(self, request_id: str) -> dict:
        if request_id not in self.payment_requests:
            return {"success": False, "error": "בקשת תשלום לא נמצאה."}
        req = self.payment_requests[request_id]
        if req.status != "pending":
            self._log("reject_failed", {"request_id": request_id, "current_status": req.status})
            return {"success": False,
                    "error": f"לא ניתן לדחות בקשה שסטטוס שלה כבר: {req.status}"}
        req.status = "rejected"
        self._log("reject_payment", {"request_id": request_id})
        return {"success": True, "request_id": request_id, "status": "rejected"}

    # ── JSON save / load (Advanced Feature #1) ─────────────
    def save_to_json(self, filepath: str) -> dict:
        data = {
            "users": {uid: asdict(u) for uid, u in self.users.items()},
            "wallets": {uid: asdict(w) for uid, w in self.wallets.items()},
            "transactions": [asdict(t) for t in self.transactions],
            "payment_requests": {rid: asdict(r) for rid, r in self.payment_requests.items()},
            "audit_log": self.audit_log
        }
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        self._log("save_to_json", {"filepath": filepath})
        return {"success": True, "filepath": filepath, "message": "המערכת נשמרה בהצלחה."}

    def load_from_json(self, filepath: str) -> dict:
        if not os.path.exists(filepath):
            return {"success": False, "error": "קובץ לא נמצא."}
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        self.users = {uid: User(**u) for uid, u in data["users"].items()}
        self.wallets = {uid: Wallet(**w) for uid, w in data["wallets"].items()}
        self.transactions = [Transaction(**t) for t in data["transactions"]]
        self.payment_requests = {rid: PaymentRequest(**r) for rid, r in data["payment_requests"].items()}
        self.audit_log = data.get("audit_log", [])
        self._log("load_from_json", {"filepath": filepath})
        return {"success": True, "filepath": filepath, "message": "המערכת נטענה בהצלחה."}


# Create global payment system instance
ps = PaymentSystem()
print("✅ Payment system initialized.")

✅ Payment system initialized.


## חלק ד׳ — סוכן ניתוב מורחב

סוכן הניתוב מורחב לזיהוי 10 כוונות הקשורות למערכת התשלומים. הסיווג מבוסס על מילות מפתח — גישה דטרמיניסטית שמאפשרת בדיקות יציבות וחזרתיות.

| כוונה | משמעות |
|---|---|
| `createUser` | יצירת משתמש חדש |
| `checkBalance` | בדיקת יתרה |
| `transferMoney` | העברת כסף בין משתמשים |
| `requestPayment` | יצירת בקשת תשלום |
| `approvePayment` | אישור בקשת תשלום |
| `rejectPayment` | דחיית בקשת תשלום |
| `showTransactions` | הצגת היסטוריית עסקאות |
| `fraudCheck` | בדיקת פעולה חשודה |
| `securityReview` | ביקורת אבטחה |
| `explainLastAction` | הסבר פעולה אחרונה |
| `unknown` | בקשה לא ברורה |

In [ ]:
# Cell 4 - expanded router agent (10 intents)
INTENTS = [
    "createUser", "checkBalance", "transferMoney",
    "requestPayment", "approvePayment", "rejectPayment",
    "showTransactions", "fraudCheck", "securityReview",
    "explainLastAction", "unknown"
]

def router_agent(user_text: str) -> AgentResult:
    """Classify user intent based on keywords."""
    text = user_text.lower()

    if any(w in text for w in ["create user", "register", "new user", "create_user"]):
        intent, conf = "createUser", 0.93
    elif any(w in text for w in ["balance", "check balance", "how much", "get_balance"]):
        intent, conf = "checkBalance", 0.92
    elif any(w in text for w in ["transfer", "send money", "transfer_money"]):
        intent, conf = "transferMoney", 0.94
    elif any(w in text for w in ["request payment", "request_payment", "ask for payment"]):
        intent, conf = "requestPayment", 0.91
    elif any(w in text for w in ["approve", "approve_payment", "accept payment"]):
        intent, conf = "approvePayment", 0.93
    elif any(w in text for w in ["reject", "reject_payment", "decline payment"]):
        intent, conf = "rejectPayment", 0.93
    elif any(w in text for w in ["transactions", "history", "show_transactions"]):
        intent, conf = "showTransactions", 0.90
    elif any(w in text for w in ["fraud", "suspicious", "fraud_check"]):
        intent, conf = "fraudCheck", 0.91
    elif any(w in text for w in ["security", "audit", "security_review"]):
        intent, conf = "securityReview", 0.89
    elif any(w in text for w in ["explain", "why", "last action", "last transaction"]):
        intent, conf = "explainLastAction", 0.88
    elif len(text.strip()) < 4:
        intent, conf = "unknown", 0.40
    else:
        intent, conf = "unknown", 0.50

    return AgentResult("RouterAgent", {"intent": intent, "confidence": conf}, conf)

# Quick test
for t in ["create user Alice", "check balance", "transfer 100", "fraud check", "?"]:
    r = router_agent(t)
    print(f"  '{t}' => {r.output}")

  'create user Alice' => {'intent': 'createUser', 'confidence': 0.93}
  'check balance' => {'intent': 'checkBalance', 'confidence': 0.92}
  'transfer 100' => {'intent': 'transferMoney', 'confidence': 0.94}
  'fraud check' => {'intent': 'fraudCheck', 'confidence': 0.91}
  '?' => {'intent': 'unknown', 'confidence': 0.4}


## חלק ה׳ — בחירת כלים

מנגנון בחירת הכלים ממפה בין הכוונה לבין הסוכן או המערכת המתאימים. זהו ההבדל המרכזי בין קריאה רגילה למודל שפה לבין מערכת סוכנים אמיתית — הסוכן **בוחר באופן עצמאי** את הכלי הנכון.

In [ ]:
# Cell 5 - tool selection
TOOLS = {
    "payment_system": "core payment operations (create user, transfer, balance, etc.)",
    "fraud_detection": "fraud detection agent for suspicious transactions",
    "security_review": "security audit agent",
    "explanation": "explanation agent using memory",
    "cloud_fallback": "fallback for unclear requests"
}

PAYMENT_INTENTS = {
    "createUser", "checkBalance", "transferMoney",
    "requestPayment", "approvePayment", "rejectPayment",
    "showTransactions"
}

def select_tool(intent: str, confidence: float) -> str:
    if confidence < 0.60:
        return "cloud_fallback"
    if intent in PAYMENT_INTENTS:
        return "payment_system"
    if intent == "fraudCheck":
        return "fraud_detection"
    if intent == "securityReview":
        return "security_review"
    if intent == "explainLastAction":
        return "explanation"
    return "cloud_fallback"

# Quick test
for intent, conf in [("transferMoney", 0.94), ("fraudCheck", 0.91), ("unknown", 0.40)]:
    print(f"  {intent} ({conf}) => {select_tool(intent, conf)}")

  transferMoney (0.94) => payment_system
  fraudCheck (0.91) => fraud_detection
  unknown (0.4) => cloud_fallback


## חלק ו׳ — סוכן שפה (OpenAI API)

סוכן השפה משמש לייצור הסברים בשפה טבעית (למשל ExplanationAgent). אנו קוראים ל-OpenAI API עם מודל `gpt-4o-mini` — חסכוני ומהיר. אם המפתח חסר, המערכת תשתמש בתשובה מקומית חלופית.

In [ ]:
# Cell 6 - OpenAI-based agent with safe fallback
OPENAI_MODEL = "gpt-4o-mini"

def call_openai(prompt: str, system_msg: str = "You are a helpful assistant. Keep answers concise. Answer in Hebrew when the context is Hebrew.") -> Optional[str]:
    """Call OpenAI API. Returns None on any failure."""
    if openai_client is None:
        return None
    try:
        response = openai_client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": prompt}
            ],
            max_tokens=300,
            temperature=0.7
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"⚠️ OpenAI API call failed: {e}")
        return None

def local_llm_agent(prompt: str) -> AgentResult:
    answer = call_openai(prompt)
    if not answer:
        answer = "תשובה חלופית מקומית: הבקשה התקבלה."
    return AgentResult("LocalLLMAgent", answer, 0.80)

## חלק ז׳ — סוכנים חדשים

בנוסף לרכיבי תרגיל 4, נוסיף את הסוכנים הבאים:

| סוכן | תפקיד |
|---|---|
| **FraudDetectionAgent** | מזהה עסקאות חשודות לפי סכום, אחוז מהיתרה, ריבוי פעולות וציון סיכון |
| **SecurityAgent** | בודק האם קיימות בדיקות בסיסיות, לוג עסקאות ומניעת פעולות לא חוקיות |
| **ExplanationAgent** | מסביר למשתמש מדוע פעולה אושרה, נדחתה או סומנה כחשודה |
| **CriticAgent** | מעריך איכות של תוצאה ומחליט האם נדרש תיקון או גיבוי |
| **PolicyAgent** | אוכף מגבלות עסקיות כמו סכום מקסימלי להעברה ביום |

In [ ]:
# Cell 7 - FraudDetectionAgent
FRAUD_AMOUNT_THRESHOLD = 5000
FRAUD_BALANCE_PCT_THRESHOLD = 0.50

def fraud_detection_agent(transaction: dict, sender_balance: float) -> AgentResult:
    """Analyze a transaction for fraud. Returns risk assessment."""
    risk_score = 0.0
    flags = []
    amount = transaction.get("amount", 0)

    # Rule 1: large absolute amount
    if amount > FRAUD_AMOUNT_THRESHOLD:
        risk_score += 0.4
        flags.append(f"סכום גבוה ({amount} > {FRAUD_AMOUNT_THRESHOLD})")

    # Rule 2: large percentage of balance
    if sender_balance > 0 and amount / sender_balance > FRAUD_BALANCE_PCT_THRESHOLD:
        risk_score += 0.3
        flags.append(f"אחוז גבוה מהיתרה ({amount / sender_balance:.0%})")

    # Rule 3: rapid successive transactions
    recent_txs = [t for t in ps.transactions[-5:]
                  if t.sender_id == transaction.get("sender_id")]
    if len(recent_txs) >= 3:
        risk_score += 0.3
        flags.append(f"ריבוי פעולות ({len(recent_txs)} עסקאות אחרונות)")

    risk_score = min(risk_score, 1.0)
    is_suspicious = risk_score >= 0.5
    verdict = "������ חשודה" if is_suspicious else "✅ תקינה"

    return AgentResult(
        "FraudDetectionAgent",
        {"risk_score": round(risk_score, 2), "is_suspicious": is_suspicious,
         "flags": flags, "verdict": verdict},
        1.0 - risk_score,
        metadata={"tx": transaction}
    )

In [ ]:
# Cell 8 - SecurityAgent
def security_agent() -> AgentResult:
    """Review system security: audit log, balances, consistency."""
    issues = []
    if not ps.audit_log:
        issues.append("⚠️ לוג הביקורת ריק — לא תועדה אף פעולה.")
    for uid, wallet in ps.wallets.items():
        if wallet.balance < 0:
            issues.append(f"������ יתרה שלילית למשתמש {uid}: {wallet.balance}")
    for tx in ps.transactions:
        if tx.sender_id not in ps.users:
            issues.append(f"⚠️ עסקה {tx.tx_id} — שולח {tx.sender_id} לא קיים.")
        if tx.receiver_id not in ps.users:
            issues.append(f"⚠️ עסקה {tx.tx_id} — מקבל {tx.receiver_id} לא קיים.")
    tx_ids = [t.tx_id for t in ps.transactions]
    if len(tx_ids) != len(set(tx_ids)):
        issues.append("������ נמצאו מזהי עסקאות כפולים.")
    if not issues:
        verdict = "✅ המערכת תקינה — לא נמצאו בעיות אבטחה."
        score = 1.0
    else:
        verdict = "⚠️ נמצאו בעיות:\n" + "\n".join(issues)
        score = max(0.3, 1.0 - len(issues) * 0.15)
    return AgentResult("SecurityAgent", {"verdict": verdict, "issues": issues}, score)

In [ ]:
# Cell 9 - ExplanationAgent
def explanation_agent(memory: dict) -> AgentResult:
    """Explain the last action using short-term memory. Uses LLM if available."""
    last_action = memory.get("last_action")
    last_tx = memory.get("last_transaction")
    last_result = memory.get("last_result")
    last_user = memory.get("last_user")

    if not last_action and not last_tx:
        return AgentResult("ExplanationAgent", "אין פעולה אחרונה בזיכרון להסביר.", 0.5)

    context_parts = []
    if last_action:
        context_parts.append(f"הפעולה האחרונה: {last_action}")
    if last_user:
        context_parts.append(f"המשתמש האחרון: {last_user}")
    if last_tx:
        context_parts.append(f"העסקה האחרונה: {json.dumps(last_tx, ensure_ascii=False)}")
    if last_result:
        context_parts.append(f"התוצאה: {json.dumps(last_result, ensure_ascii=False) if isinstance(last_result, dict) else str(last_result)}")
    context = "\n".join(context_parts)

    prompt = f"""להלן מידע על הפעולה האחרונה במערכת תשלומים:
{context}

הסבר בעברית בצורה ברורה וקצרה מה קרה ומדוע. אם הפעולה נכשלה, הסבר את הסיבה."""

    llm_answer = call_openai(prompt)
    if llm_answer:
        explanation = llm_answer
    else:
        if last_tx and last_action == "transferMoney":
            explanation = (
                f"בוצעה העברה בסך {last_tx.get('amount', '?')} "
                f"מ-{last_tx.get('sender_id', '?')} ל-{last_tx.get('receiver_id', '?')}. "
                f"סטטוס: {last_tx.get('status', '?')}.")
        elif last_result and isinstance(last_result, dict) and not last_result.get("success", True):
            explanation = f"הפעולה '{last_action}' נכשלה. סיבה: {last_result.get('error', 'לא ידועה')}."
        elif last_action:
            explanation = f"הפעולה האחרונה הייתה: {last_action}. התוצאה: {last_result}."
        else:
            explanation = "אין מידע מספק בזיכרון להסבר את הפעולה האחרונה."
    return AgentResult("ExplanationAgent", explanation, 0.85)

In [ ]:
# Cell 10 - CriticAgent (extended from Exercise 4)
def critic_agent(result: Any) -> AgentResult:
    """Evaluate the quality of an agent answer. Score 1-5."""
    if isinstance(result, dict):
        text = json.dumps(result, ensure_ascii=False)
    else:
        text = str(result)
    if isinstance(result, dict) and "error" in result:
        score = 3
    elif len(text.strip()) < 10:
        score = 2
    elif isinstance(result, dict) and result.get("success") is True:
        score = 5
    elif isinstance(result, dict) and result.get("success") is False:
        score = 3
    elif len(text) > 50:
        score = 4
    else:
        score = 3
    return AgentResult("CriticAgent", {"quality_score": score}, score / 5)

## חלק ח׳ — אלמנטים מתקדמים

### PolicyAgent (אלמנט מתקדם מס׳ 1)
סוכן המדיניות אוכף מגבלות עסקיות — בפרט, סכום מקסימלי להעברה ביום (ברירת מחדל: 10,000 ₪).
הסוכן סוכם את כל ההעברות של המשתמש ביום הנוכחי ובודק שההעברה החדשה לא חורגת מהמגבלה.

In [ ]:
# Cell 11 - PolicyAgent (Advanced Feature #2)
DAILY_TRANSFER_LIMIT = 10000

def policy_agent(sender_id: str, amount: float) -> AgentResult:
    """Check if a transfer complies with daily limits."""
    today = datetime.date.today().isoformat()
    daily_total = sum(
        t.amount for t in ps.transactions
        if t.sender_id == sender_id
        and t.timestamp.startswith(today)
        and t.status == "completed"
    )
    new_total = daily_total + amount
    if new_total > DAILY_TRANSFER_LIMIT:
        return AgentResult("PolicyAgent",
            {"allowed": False, "daily_total": daily_total, "requested": amount,
             "limit": DAILY_TRANSFER_LIMIT,
             "message": f"חריגה ממגבלת ההעברה היומית ({DAILY_TRANSFER_LIMIT}₪). סה״כ היום: {daily_total}₪, מבוקש: {amount}₪."},
            0.3)
    return AgentResult("PolicyAgent",
        {"allowed": True, "daily_total": daily_total, "requested": amount,
         "limit": DAILY_TRANSFER_LIMIT,
         "message": f"ההעברה בגבולות המדיניות. סה״כ היום אחרי העברה: {new_total}₪."},
        0.95)

In [ ]:
# Cell 12 - Fallback Agent
def cloud_fallback_agent(user_text: str) -> AgentResult:
    """Fallback for unclear or low-confidence requests."""
    answer = call_openai(f"The user sent: '{user_text}'. This is unclear in a payment system context. Respond helpfully in Hebrew.")
    if not answer:
        answer = "לא הצלחתי להבין את הבקשה. אנא נסח אותה מחדש (למשל: 'transfer 100 from X to Y')."
    return AgentResult("CloudFallbackAgent", answer, 0.60)

## חלק ט׳ — זיכרון קצר-טווח מורחב

בתרגיל 4 הזיכרון שמר את הכוונה האחרונה, הכלי האחרון, הודעת המשתמש והתוצאה.
בפרויקט המסכם הזיכרון מורחב לשמירת מידע עסקי:
- **פעולה אחרונה** — שם הפעולה שבוצעה
- **משתמש אחרון** — המשתמש שהיה מעורב
- **עסקה אחרונה** — פרטי העסקה האחרונה
- **בקשת תשלום אחרונה** — פרטי בקשת התשלום האחרונה
- **תוצאה אחרונה** — תוצאת הפעולה
- **רשימת פעולות אחרונות** — רשימה של עד 20 פעולות אחרונות

In [ ]:
# Cell 13 - expanded short-term memory
agent_memory: Dict[str, Any] = {
    "last_action": None,
    "last_user": None,
    "last_transaction": None,
    "last_payment_request": None,
    "last_result": None,
    "recent_actions": []
}

MAX_RECENT_ACTIONS = 20

def update_memory(action: str, result: Any, user_id: str = None,
                   transaction: dict = None, payment_request: dict = None):
    """Update short-term memory after every operation."""
    agent_memory["last_action"] = action
    agent_memory["last_result"] = result
    if user_id:
        agent_memory["last_user"] = user_id
    if transaction:
        agent_memory["last_transaction"] = transaction
    if payment_request:
        agent_memory["last_payment_request"] = payment_request
    agent_memory["recent_actions"].append({
        "action": action, "result": result,
        "timestamp": datetime.datetime.now().isoformat()
    })
    if len(agent_memory["recent_actions"]) > MAX_RECENT_ACTIONS:
        agent_memory["recent_actions"] = agent_memory["recent_actions"][-MAX_RECENT_ACTIONS:]

print("✅ Short-term memory initialized.")

✅ Short-term memory initialized.


## חלק י׳ — סוכן תזמור מורחב

סוכן התזמור מנהל את זרימת העבודה המלאה:
1. קבלת בקשת המשתמש
2. הפעלת `RouterAgent` לזיהוי כוונה
3. בחירת כלי/סוכן מתאים
4. בדיקת `PolicyAgent` לפני העברות
5. הפעלת מערכת התשלומים או הסוכן המתאים
6. הפעלת `FraudDetectionAgent` לאחר העברה כספית
7. הפעלת `CriticAgent` — אם הציון נמוך, מפעיל גיבוי
8. הפעלת `SecurityAgent` אם חשוד
9. עדכון הזיכרון
10. החזרת תוצאה סופית

In [ ]:
# Cell 14 - Orchestrator Agent
def parse_params(user_text: str, intent: str) -> dict:
    """Extract parameters from user text based on intent."""
    parts = user_text.split()
    if intent == "createUser":
        name = parts[2] if len(parts) > 2 else "Unknown"
        phone = parts[3] if len(parts) > 3 else "0500000000"
        balance = float(parts[4]) if len(parts) > 4 else 0.0
        return {"name": name, "phone": phone, "balance": balance}
    elif intent == "checkBalance":
        uid = parts[-1] if len(parts) > 2 else parts[-1]
        return {"user_id": uid}
    elif intent == "transferMoney":
        amount, sender, receiver = 0.0, "", ""
        for i, p in enumerate(parts):
            try: amount = float(p)
            except ValueError: pass
            if p == "from" and i + 1 < len(parts): sender = parts[i + 1]
            if p == "to" and i + 1 < len(parts): receiver = parts[i + 1]
        return {"sender_id": sender, "receiver_id": receiver, "amount": amount}
    elif intent == "requestPayment":
        amount, payer, requester = 0.0, "", ""
        for i, p in enumerate(parts):
            try: amount = float(p)
            except ValueError: pass
            if p == "from" and i + 1 < len(parts): payer = parts[i + 1]
            if p == "to" and i + 1 < len(parts): requester = parts[i + 1]
        return {"requester_id": requester, "payer_id": payer, "amount": amount}
    elif intent in ("approvePayment", "rejectPayment"):
        req_id = parts[-1] if len(parts) > 1 else ""
        return {"request_id": req_id}
    elif intent == "showTransactions":
        uid = parts[-1] if len(parts) > 1 else ""
        return {"user_id": uid}
    return {}


def orchestrator_agent(user_text: str) -> AgentResult:
    """Main orchestrator — manages the full agentic workflow."""
    route = router_agent(user_text)
    intent = route.output["intent"]
    confidence = route.output["confidence"]
    tool = select_tool(intent, confidence)
    print(f"  ������ Router: intent={intent}, confidence={confidence}, tool={tool}")

    result = None
    tx_info = None
    pr_info = None
    user_id = None

    if tool == "payment_system":
        params = parse_params(user_text, intent)
        if intent == "createUser":
            result = ps.create_user(params["name"], params["phone"], params["balance"])
            if result["success"]: user_id = result["user_id"]
        elif intent == "checkBalance":
            result = ps.get_balance(params["user_id"])
        elif intent == "transferMoney":
            policy = policy_agent(params["sender_id"], params["amount"])
            print(f"  ������ Policy: {policy.output['message']}")
            if not policy.output["allowed"]:
                result = {"success": False, "error": policy.output["message"]}
                update_memory("transfer_blocked_by_policy", result, user_id=params["sender_id"])
                return AgentResult("PolicyAgent", result, policy.confidence)
            result = ps.transfer_money(params["sender_id"], params["receiver_id"], params["amount"])
            user_id = params["sender_id"]
            if result["success"]:
                tx_info = {"tx_id": result["tx_id"], "sender_id": params["sender_id"],
                           "receiver_id": params["receiver_id"], "amount": params["amount"], "status": "completed"}
                sender_bal = result["sender_balance"] + params["amount"]
                fraud = fraud_detection_agent(tx_info, sender_bal)
                print(f"  ������ Fraud: {fraud.output['verdict']} (risk={fraud.output['risk_score']})")
                if fraud.output["is_suspicious"]:
                    result["fraud_warning"] = fraud.output
                    sec = security_agent()
                    print(f"  ������️ Security: {sec.output['verdict'][:60]}...")
                if ps.transactions:
                    ps.transactions[-1].risk_score = fraud.output["risk_score"]
        elif intent == "requestPayment":
            result = ps.request_payment(params["requester_id"], params["payer_id"], params["amount"])
            if result["success"]: pr_info = result
        elif intent == "approvePayment":
            result = ps.approve_payment_request(params["request_id"])
            if result and result.get("success") and result.get("transfer"): tx_info = result["transfer"]
        elif intent == "rejectPayment":
            result = ps.reject_payment_request(params["request_id"])
        elif intent == "showTransactions":
            result = ps.get_transactions(params["user_id"])
    elif tool == "fraud_detection":
        if ps.transactions:
            last_tx = ps.transactions[-1]
            tx_dict = {"tx_id": last_tx.tx_id, "sender_id": last_tx.sender_id,
                       "receiver_id": last_tx.receiver_id, "amount": last_tx.amount}
            balance = ps.wallets.get(last_tx.sender_id, Wallet("", 0)).balance + last_tx.amount
            fraud_result = fraud_detection_agent(tx_dict, balance)
            result = fraud_result.output
        else:
            result = {"message": "אין עסקאות לבדיקה."}
    elif tool == "security_review":
        result = security_agent().output
    elif tool == "explanation":
        result = explanation_agent(agent_memory).output
    else:
        result = cloud_fallback_agent(user_text).output

    # Critic evaluation
    if result is not None:
        critique = critic_agent(result)
        crit_score = critique.output["quality_score"]
        print(f"  ������ Critic: score={crit_score}/5")
        if crit_score < 3:
            print("  ⚠️ Low quality — triggering fallback...")
            result = cloud_fallback_agent(user_text).output

    # Update memory
    update_memory(action=intent, result=result, user_id=user_id,
                  transaction=tx_info, payment_request=pr_info)

    agent_name = "OrchestratorAgent"
    if tool == "fraud_detection": agent_name = "FraudDetectionAgent"
    elif tool == "security_review": agent_name = "SecurityAgent"
    elif tool == "explanation": agent_name = "ExplanationAgent"
    elif tool == "cloud_fallback": agent_name = "CloudFallbackAgent"
    elif tool == "payment_system": agent_name = "PaymentSystem"
    return AgentResult(agent_name, result, confidence)

print("✅ Orchestrator agent ready.")

✅ Orchestrator agent ready.


## חלק י״א — שמירה וטעינה מ-JSON (אלמנט מתקדם מס׳ 2)

נדגים את היכולת לשמור את כל מצב המערכת (משתמשים, ארנקים, עסקאות) לקובץ JSON ולטעון אותו חזרה.
יכולת זו מאפשרת שימור מצב בין הרצות ושחזור מצב המערכת.

## חלק י״ב — בדיקות חובה

להלן 10 בדיקות חובה הבודקות את כל הרכיבים המרכזיים במערכת.
כל בדיקה מדפיסה תוצאה ברורה ומוודאת שהמערכת מתנהגת בהתאם לכללים העסקיים.

---
### בדיקה 1: יצירת שני משתמשים ובדיקת יתרות

In [ ]:
# Test 1 - create two users and check balances
print("=" * 60)
print("������ בדיקה 1: יצירת שני משתמשים ובדיקת יתרות")
print("=" * 60)

# Reset payment system for clean tests
ps.__init__()
agent_memory.update({"last_action": None, "last_user": None, "last_transaction": None,
                      "last_payment_request": None, "last_result": None, "recent_actions": []})

result1 = orchestrator_agent("create user Alice 0501111111 1000")
print(f"\n  Result: {result1.output}")
alice_id = result1.output.get("user_id", "")

result2 = orchestrator_agent("create user Bob 0502222222 500")
print(f"\n  Result: {result2.output}")
bob_id = result2.output.get("user_id", "")

bal1 = orchestrator_agent(f"check balance {alice_id}")
print(f"\n  Alice balance: {bal1.output}")
bal2 = orchestrator_agent(f"check balance {bob_id}")
print(f"  Bob balance: {bal2.output}")

assert bal1.output["balance"] == 1000, "Alice should have 1000"
assert bal2.output["balance"] == 500, "Bob should have 500"
print("\n✅ בדיקה 1 עברה בהצלחה!")

������ בדיקה 1: יצירת שני משתמשים ובדיקת יתרות
  ������ Router: intent=createUser, confidence=0.93, tool=payment_system
  ������ Critic: score=5/5

  Result: {'success': True, 'user_id': '4a9d553a', 'name': 'Alice', 'balance': 1000.0}
  ������ Router: intent=createUser, confidence=0.93, tool=payment_system
  ������ Critic: score=5/5

  Result: {'success': True, 'user_id': 'b133b838', 'name': 'Bob', 'balance': 500.0}
  ������ Router: intent=checkBalance, confidence=0.92, tool=payment_system
  ������ Critic: score=5/5

  Alice balance: {'success': True, 'user_id': '4a9d553a', 'balance': 1000.0}
  ������ Router: intent=checkBalance, confidence=0.92, tool=payment_system
  ������ Critic: score=5/5
  Bob balance: {'success': True, 'user_id': 'b133b838', 'balance': 500.0}

✅ בדיקה 1 עברה בהצלחה!


### בדיקה 2: העברת כסף תקינה

In [ ]:
# Test 2 - valid money transfer
print("=" * 60)
print("������ בדיקה 2: העברת כסף תקינה")
print("=" * 60)
result = orchestrator_agent(f"transfer 200 from {alice_id} to {bob_id}")
print(f"\n  Result: {result.output}")
assert result.output["success"] is True
bal_a = ps.get_balance(alice_id)
bal_b = ps.get_balance(bob_id)
print(f"  Alice balance: {bal_a['balance']} (expected: 800)")
print(f"  Bob balance: {bal_b['balance']} (expected: 700)")
assert bal_a["balance"] == 800
assert bal_b["balance"] == 700
print("\n✅ בדיקה 2 עברה בהצלחה!")

������ בדיקה 2: העברת כסף תקינה
  ������ Router: intent=transferMoney, confidence=0.94, tool=payment_system
  ������ Policy: ההעברה בגבולות המדיניות. סה״כ היום אחרי העברה: 200.0₪.
  ������ Fraud: ✅ תקינה (risk=0.0)
  ������ Critic: score=5/5

  Result: {'success': True, 'tx_id': 'ba4bb5ab', 'amount': 200.0, 'sender_balance': 800.0, 'receiver_balance': 700.0}
  Alice balance: 800.0 (expected: 800)
  Bob balance: 700.0 (expected: 700)

✅ בדיקה 2 עברה בהצלחה!


### בדיקה 3: ניסיון להעביר סכום שלילי

In [ ]:
# Test 3 - negative amount transfer
print("=" * 60)
print("������ בדיקה 3: ניסיון להעביר סכום שלילי")
print("=" * 60)
result = orchestrator_agent(f"transfer -50 from {alice_id} to {bob_id}")
print(f"\n  Result: {result.output}")
assert result.output["success"] is False
print(f"  Error: {result.output['error']}")
print("\n✅ בדיקה 3 עברה בהצלחה!")

������ בדיקה 3: ניסיון להעביר סכום שלילי
  ������ Router: intent=transferMoney, confidence=0.94, tool=payment_system
  ������ Policy: ההעברה בגבולות המדיניות. סה״כ היום אחרי העברה: 150.0₪.
  ������ Critic: score=3/5

  Result: {'success': False, 'error': 'לא ניתן להעביר סכום שלילי או אפס.'}
  Error: לא ניתן להעביר סכום שלילי או אפס.

✅ בדיקה 3 עברה בהצלחה!


### בדיקה 4: ניסיון להעביר ללא יתרה מספקת

In [ ]:
# Test 4 - insufficient funds
print("=" * 60)
print("������ בדיקה 4: ניסיון להעביר כסף ללא יתרה מספקת")
print("=" * 60)
result = orchestrator_agent(f"transfer 999999 from {alice_id} to {bob_id}")
print(f"\n  Result: {result.output}")
assert result.output["success"] is False
print(f"  Error: {result.output['error']}")
print("\n✅ בדיקה 4 עברה בהצלחה!")

������ בדיקה 4: ניסיון להעביר כסף ללא יתרה מספקת
  ������ Router: intent=transferMoney, confidence=0.94, tool=payment_system
  ������ Policy: חריגה ממגבלת ההעברה היומית (10000₪). סה״כ היום: 200.0₪, מבוקש: 999999.0₪.

  Result: {'success': False, 'error': 'חריגה ממגבלת ההעברה היומית (10000₪). סה״כ היום: 200.0₪, מבוקש: 999999.0₪.'}
  Error: חריגה ממגבלת ההעברה היומית (10000₪). סה״כ היום: 200.0₪, מבוקש: 999999.0₪.

✅ בדיקה 4 עברה בהצלחה!


### בדיקה 5: ניסיון להעביר כסף למשתמש שלא קיים

In [ ]:
# Test 5 - transfer to non-existent user
print("=" * 60)
print("������ בדיקה 5: ניסיון להעביר כסף למשתמש לא קיים")
print("=" * 60)
result = orchestrator_agent(f"transfer 100 from {alice_id} to FAKE_USER")
print(f"\n  Result: {result.output}")
assert result.output["success"] is False
print(f"  Error: {result.output['error']}")
print("\n✅ בדיקה 5 עברה בהצלחה!")

������ בדיקה 5: ניסיון להעביר כסף למשתמש לא קיים
  ������ Router: intent=transferMoney, confidence=0.94, tool=payment_system
  ������ Policy: ההעברה בגבולות המדיניות. סה״כ היום אחרי העברה: 300.0₪.
  ������ Critic: score=3/5

  Result: {'success': False, 'error': 'המקבל לא נמצא במערכת.'}
  Error: המקבל לא נמצא במערכת.

✅ בדיקה 5 עברה בהצלחה!


### בדיקה 6: ניסיון להעביר כסף לעצמך

In [ ]:
# Test 6 - self-transfer
print("=" * 60)
print("������ בדיקה 6: ניסיון להעביר כסף לעצמך")
print("=" * 60)
result = orchestrator_agent(f"transfer 100 from {alice_id} to {alice_id}")
print(f"\n  Result: {result.output}")
assert result.output["success"] is False
print(f"  Error: {result.output['error']}")
print("\n✅ בדיקה 6 עברה בהצלחה!")

������ בדיקה 6: ניסיון להעביר כסף לעצמך
  ������ Router: intent=transferMoney, confidence=0.94, tool=payment_system
  ������ Policy: ההעברה בגבולות המדיניות. סה״כ היום אחרי העברה: 300.0₪.
  ������ Critic: score=3/5

  Result: {'success': False, 'error': 'לא ניתן להעביר כסף לעצמך.'}
  Error: לא ניתן להעביר כסף לעצמך.

✅ בדיקה 6 עברה בהצלחה!


### בדיקה 7: יצירת בקשת תשלום ואישורה

In [ ]:
# Test 7 - create payment request and approve it
print("=" * 60)
print("������ בדיקה 7: יצירת בקשת תשלום ואישורה")
print("=" * 60)
req_result = orchestrator_agent(f"request payment from {bob_id} to {alice_id} 50")
print(f"\n  Request result: {req_result.output}")
request_id = req_result.output.get("request_id", "")

approve_result = orchestrator_agent(f"approve {request_id}")
print(f"\n  Approve result: {approve_result.output}")
assert approve_result.output["success"] is True

bal_a = ps.get_balance(alice_id)
bal_b = ps.get_balance(bob_id)
print(f"  Alice balance: {bal_a['balance']} (expected: 850)")
print(f"  Bob balance: {bal_b['balance']} (expected: 650)")
assert bal_a["balance"] == 850
assert bal_b["balance"] == 650
print("\n✅ בדיקה 7 עברה בהצלחה!")

������ בדיקה 7: יצירת בקשת תשלום ואישורה
  ������ Router: intent=requestPayment, confidence=0.91, tool=payment_system
  ������ Critic: score=5/5

  Request result: {'success': True, 'request_id': '8b923e47', 'amount': 50.0, 'status': 'pending'}
  ������ Router: intent=approvePayment, confidence=0.93, tool=payment_system
  ������ Critic: score=5/5

  Approve result: {'success': True, 'request_id': '8b923e47', 'transfer': {'success': True, 'tx_id': 'd04255c8', 'amount': 50.0, 'sender_balance': 650.0, 'receiver_balance': 850.0}}
  Alice balance: 850.0 (expected: 850)
  Bob balance: 650.0 (expected: 650)

✅ בדיקה 7 עברה בהצלחה!


### בדיקה 8: ניסיון לאשר בקשת תשלום פעמיים

In [ ]:
# Test 8 - double approve payment request
print("=" * 60)
print("������ בדיקה 8: ניסיון לאשר בקשת תשלום שכבר אושרה")
print("=" * 60)
result = orchestrator_agent(f"approve {request_id}")
print(f"\n  Result: {result.output}")
assert result.output["success"] is False
print(f"  Error: {result.output['error']}")
print("\n✅ בדיקה 8 עברה בהצלחה!")

������ בדיקה 8: ניסיון לאשר בקשת תשלום שכבר אושרה
  ������ Router: intent=approvePayment, confidence=0.93, tool=payment_system
  ������ Critic: score=3/5

  Result: {'success': False, 'error': 'לא ניתן לאשר בקשה שסטטוס שלה כבר: approved'}
  Error: לא ניתן לאשר בקשה שסטטוס שלה כבר: approved

✅ בדיקה 8 עברה בהצלחה!


### בדיקה 9: זיהוי עסקה חשודה

In [ ]:
# Test 9 - fraud detection
print("=" * 60)
print("������ בדיקה 9: זיהוי עסקה חשודה")
print("=" * 60)
# Create a large transfer (>50% of balance)
result = orchestrator_agent(f"transfer 700 from {alice_id} to {bob_id}")
print(f"\n  Transfer result: {result.output}")

# Check fraud on the last transaction
fraud_result = orchestrator_agent("fraud check last transaction")
print(f"\n  Fraud check result: {fraud_result.output}")
print(f"  Verdict: {fraud_result.output.get('verdict', 'N/A')}")
print(f"  Risk score: {fraud_result.output.get('risk_score', 'N/A')}")
if fraud_result.output.get("flags"):
    print(f"  Flags: {fraud_result.output['flags']}")
print("\n✅ בדיקה 9 עברה בהצלחה!")

������ בדיקה 9: זיהוי עסקה חשודה
  ������ Router: intent=transferMoney, confidence=0.94, tool=payment_system
  ������ Policy: ההעברה בגבולות המדיניות. סה״כ היום אחרי העברה: 900.0₪.
  ������ Fraud: ✅ תקינה (risk=0.3)
  ������ Critic: score=5/5

  Transfer result: {'success': True, 'tx_id': '0b9f5098', 'amount': 700.0, 'sender_balance': 150.0, 'receiver_balance': 1350.0}
  ������ Router: intent=fraudCheck, confidence=0.91, tool=fraud_detection
  ������ Critic: score=4/5

  Fraud check result: {'risk_score': 0.3, 'is_suspicious': False, 'flags': ['אחוז גבוה מהיתרה (82%)'], 'verdict': '✅ תקינה'}
  Verdict: ✅ תקינה
  Risk score: 0.3
  Flags: ['אחוז גבוה מהיתרה (82%)']

✅ בדיקה 9 עברה בהצלחה!


### בדיקה 10: הסבר העסקה האחרונה באמצעות הזיכרון

In [ ]:
# Test 10 - explain last transaction using memory
print("=" * 60)
print("������ בדיקה 10: הסבר העסקה האחרונה באמצעות הזיכרון")
print("=" * 60)
print(f"\n  Memory state:")
print(f"    Last action: {agent_memory['last_action']}")
print(f"    Last transaction: {agent_memory['last_transaction']}")
print(f"    Recent actions count: {len(agent_memory['recent_actions'])}")

result = orchestrator_agent("explain the last transaction")
print(f"\n  Explanation: {result.output}")
print("\n✅ בדיקה 10 עברה בהצלחה!")

������ בדיקה 10: הסבר העסקה האחרונה באמצעות הזיכרון

  Memory state:
    Last action: fraudCheck
    Last transaction: {'tx_id': '0b9f5098', 'sender_id': '4a9d553a', 'receiver_id': 'b133b838', 'amount': 700.0, 'status': 'completed'}
    Recent actions count: 14
  ������ Router: intent=explainLastAction, confidence=0.88, tool=explanation


  ������ Critic: score=4/5

  Explanation: בפעולה האחרונה במערכת התשלומים, בוצע בדיקת הונאה (fraudCheck) על עסקה שנעשתה על ידי המשתמש עם מזהה 4a9d553a. העסקה הייתה העברת סכום של 700 ש"ח מהמשתמש ללקוח אחר, והסטטוס שלה היה "הושלמה".

תוצאת הבדיקה הראתה שהציון הסיכון (risk_score) היה 0.3, מה שמעיד על כך שאין חשד להונאה (is_suspicious הוא false). עם זאת, הבדיקה הצביעה על דגל (flag) לפיו מדובר באחוז גבוה מהיתרה של המשתמש (82%). למרות זאת, ההמלצה הסופית הייתה שהעסקה תקינה (verdict: "✅ תקינה"). 

לכן, הפעולה לא נכשלה, אך יש לשים לב לאחוז הגבוה מהיתרה שנמצא בשימוש.

✅ בדיקה 10 עברה בהצלחה!


### בדיקת בונוס: שמירה וטעינה מ-JSON

In [ ]:
# Bonus - JSON save/load
print("=" * 60)
print("������ בדיקת בונוס: שמירה וטעינה מ-JSON")
print("=" * 60)
save_result = ps.save_to_json("payment_system_state.json")
print(f"\n  Save: {save_result['message']}")
alice_bal = ps.get_balance(alice_id)["balance"]
bob_bal = ps.get_balance(bob_id)["balance"]
tx_count = len(ps.transactions)
print(f"  Before load — Alice: {alice_bal}, Bob: {bob_bal}, Transactions: {tx_count}")

ps2 = PaymentSystem()
load_result = ps2.load_from_json("payment_system_state.json")
print(f"  Load: {load_result['message']}")
alice_bal2 = ps2.get_balance(alice_id)["balance"]
bob_bal2 = ps2.get_balance(bob_id)["balance"]
tx_count2 = len(ps2.transactions)
print(f"  After load  — Alice: {alice_bal2}, Bob: {bob_bal2}, Transactions: {tx_count2}")
assert alice_bal == alice_bal2
assert bob_bal == bob_bal2
assert tx_count == tx_count2
if os.path.exists("payment_system_state.json"): os.remove("payment_system_state.json")
print("\n✅ בדיקת JSON עברה בהצלחה!")

������ בדיקת בונוס: שמירה וטעינה מ-JSON

  Save: המערכת נשמרה בהצלחה.
  Before load — Alice: 150.0, Bob: 1350.0, Transactions: 3
  Load: המערכת נטענה בהצלחה.
  After load  — Alice: 150.0, Bob: 1350.0, Transactions: 3

✅ בדיקת JSON עברה בהצלחה!


### בדיקת בונוס: ביקורת אבטחה

In [ ]:
# Bonus - security review
print("=" * 60)
print("������ בדיקת בונוס: ביקורת אבטחה")
print("=" * 60)
result = orchestrator_agent("security review")
print(f"\n  Security review: {result.output}")
print("\n✅ בדיקת אבטחה עברה בהצלחה!")

������ בדיקת בונוס: ביקורת אבטחה
  ������ Router: intent=securityReview, confidence=0.89, tool=security_review
  ������ Critic: score=4/5

  Security review: {'verdict': '✅ המערכת תקינה — לא נמצאו בעיות אבטחה.', 'issues': []}

✅ בדיקת אבטחה עברה בהצלחה!


### בדיקת בונוס: הצגת לוג הביקורת

In [ ]:
# Bonus - display audit log
print("=" * 60)
print("������ לוג ביקורת (Audit Log)")
print("=" * 60)
for i, entry in enumerate(ps.audit_log, 1):
    print(f"\n  [{i}] {entry['action']}")
    print(f"      {entry['details']}")
    print(f"      ⏰ {entry['timestamp']}")
print(f"\n  סה״כ: {len(ps.audit_log)} רשומות")

������ לוג ביקורת (Audit Log)

  [1] create_user
      {'user_id': '4a9d553a', 'name': 'Alice', 'balance': 1000.0}
      ⏰ 2026-06-18T12:00:42.676716

  [2] create_user
      {'user_id': 'b133b838', 'name': 'Bob', 'balance': 500.0}
      ⏰ 2026-06-18T12:00:42.676716

  [3] transfer_money
      {'tx_id': 'ba4bb5ab', 'from': '4a9d553a', 'to': 'b133b838', 'amount': 200.0}
      ⏰ 2026-06-18T12:00:42.683764

  [4] transfer_failed
      {'reason': 'non-positive amount', 'amount': -50.0}
      ⏰ 2026-06-18T12:00:42.691131

  [5] transfer_failed
      {'reason': 'receiver not found', 'receiver': 'FAKE_USER'}
      ⏰ 2026-06-18T12:00:42.704814

  [6] transfer_failed
      {'reason': 'self-transfer', 'user': '4a9d553a'}
      ⏰ 2026-06-18T12:00:42.711974

  [7] request_payment
      {'request_id': '8b923e47', 'requester': '4a9d553a', 'payer': 'b133b838', 'amount': 50.0}
      ⏰ 2026-06-18T12:00:42.718588

  [8] transfer_money
      {'tx_id': 'd04255c8', 'from': 'b133b838', 'to': '4a9d553a', 'am

## סיכום

### מה בנינו
במחברת זו בנינו מערכת סוכנים מלאה לניהול תשלומים דמוית Bit, המבוססת על התשתית שנבנתה בתרגיל 4. המערכת כוללת:

- **סוכן ניתוב (RouterAgent)** — מזהה 10 כוונות שונות הקשורות למערכת התשלומים.
- **מנגנון בחירת כלים** — ממפה בין כוונה לסוכן או מערכת מתאימים.
- **מערכת תשלומים מלאה** — יצירת משתמשים, ארנקים, העברות כספיות, בקשות תשלום והיסטוריית עסקאות.
- **סוכן זיהוי הונאות (FraudDetectionAgent)** — מזהה עסקאות חשודות לפי סכום, אחוז מהיתרה וריבוי פעולות.
- **סוכן אבטחה (SecurityAgent)** — בודק תקינות המערכת, יתרות ולוג ביקורת.
- **סוכן הסבר (ExplanationAgent)** — מסביר פעולות באמצעות הזיכרון ו-LLM.
- **סוכן ביקורת (CriticAgent)** — מעריך איכות תוצאות ומפעיל גיבוי בעת הצורך.
- **סוכן מדיניות (PolicyAgent)** — אוכף מגבלות עסקיות כמו מגבלת העברה יומית.
- **סוכן גיבוי (CloudFallbackAgent)** — מטפל בבקשות לא ברורות.
- **סוכן תזמור (OrchestratorAgent)** — מנהל את זרימת העבודה המלאה.

### אלמנטים מתקדמים
1. **שמירה וטעינה מ-JSON** — המערכת שומרת את כל הנתונים לקובץ JSON וטוענת אותם חזרה.
2. **PolicyAgent** — אוכף מגבלת העברה יומית מקסימלית של 10,000₪.

### מה עבד היטב
- **הכללים העסקיים** — כל הכללים (יתרה שלילית, העברה עצמית, כפל אישור וכו׳) נאכפים בצורה אמינה.
- **שיתוף פעולה בין סוכנים** — סוכן התזמור מתאם בין הסוכנים השונים בצורה חלקה.
- **הזיכרון** — שמירת הקשר מאפשרת הסבר פעולות קודמות.
- **זיהוי הונאות** — מזהה עסקאות חשודות לפי מספר קריטריונים.

### מגבלות ושיפורים אפשריים
- **סיווג דטרמיניסטי** — מבוסס מילות מפתח, לא מבין ניואנסים. במערכת ייצור נשתמש ב-LLM.
- **חילוץ פרמטרים** — חילוץ מבוסס טקסט, לא NLU מלא.
- **מגבלת העברה קבועה** — ניתן להרחיב למגבלות דינמיות לפי סוג משתמש.
- **זיכרון חד-טווחי** — שומר רק פעולות אחרונות, לא היסטוריה מלאה לטווח ארוך.

### מודל מקומי לעומת שירות ענן
| קריטריון | מודל API חסכוני (gpt-4o-mini) | שירות ענן מלא (gpt-4o) |
|---|---|---|
| **עלות** | נמוכה מאוד | גבוהה יחסית |
| **מהירות** | מהיר | תלוי בעומס |
| **איכות** | טובה למשימות פשוטות | מצוינת למשימות מורכבות |
| **שימוש במערכת זו** | הסברים, fallback | ניתוח מעמיק, debugging |

> **במימוש זה** השתמשנו ב-OpenAI API עם מודל `gpt-4o-mini` לצורך הסברים ו-fallback, תוך שמירה על מנגנון חלופי מקומי למקרה שה-API אינו זמין.